# D8: Middle-Bridge Replacement — Chaining/Redundancy Control

Tests whether disrupting the middle 20 tokens (d=51–70) changes the
marginal influence of preserved near (d=1–50) and far (d=71–100) context.

If near/far change even though their tokens are unchanged → chaining, not additive.

Conditions:
- D8a: shuffle original middle tokens
- D8b: same-topic foreign middle
- D8c: different-topic foreign middle
- D8d: same-document distant middle

wiki_zh + wiki_ja, Llama, 1 shuffle, corrected marginals.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate sentence-transformers

import numpy as np
import json, math, os, gc, random, time, shutil
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_D8')
BASE.mkdir(parents=True, exist_ok=True)
FORMAL = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_formal')
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
N_SHUFFLES = 1
SEED = 20260429

# Band definitions (in distance-from-target units)
NEAR = (1, 50)    # d=1–50, closest to target
BRIDGE = (51, 70) # d=51–70, middle bridge
FAR = (71, 100)   # d=71–100, most distant

# In temporal-order list (oldest to newest):
# ctx[0..29] = far (d=100..71)
# ctx[30..49] = bridge (d=70..51)
# ctx[50..99] = near (d=50..1)
FAR_SLICE = slice(0, 30)      # far band tokens
BRIDGE_SLICE = slice(30, 50)   # bridge tokens
NEAR_SLICE = slice(50, 100)    # near band tokens

# Copy intact from formal run
for cn in CORPORA:
    src = FORMAL / f'llama_{cn}_intact.json'
    dst = BASE / f'llama_{cn}_intact.json'
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f'Copied {cn} intact')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Bands: near d={NEAR}, bridge d={BRIDGE}, far d={FAR}')
print('Setup done')

In [ ]:
# === Build topic-matching index for D8b/D8c donors ===
from sentence_transformers import SentenceTransformer

print('Loading sentence-transformer for topic matching...')
st_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

def build_donor_index(corpus_path, tokenizer):
    """Build paragraph embeddings for donor selection."""
    docs = []
    with open(corpus_path) as f:
        for line in f: docs.append(json.loads(line))
    
    paragraphs = []  # (doc_idx, doc_id, token_start, token_end, tokens, text)
    for di, doc in enumerate(docs):
        toks = tokenizer.encode(doc['text'], add_special_tokens=False)
        # Chunk into ~100-token paragraphs
        pos = 0
        while pos < len(toks):
            end = min(pos + 100, len(toks))
            if end - pos >= 20:
                text = tokenizer.decode(toks[pos:end])
                paragraphs.append((di, doc.get('doc_id',''), pos, end, toks[pos:end], text))
            pos = end
    
    texts = [p[5] for p in paragraphs]
    embeddings = st_model.encode(texts, show_progress_bar=False, batch_size=64)
    
    # Per-document mean embeddings
    doc_embs = {}
    for p, emb in zip(paragraphs, embeddings):
        doc_embs.setdefault(p[1], []).append(emb)
    doc_mean_embs = {did: np.mean(embs, axis=0) for did, embs in doc_embs.items()}
    
    return docs, paragraphs, embeddings, doc_mean_embs

print('Topic matcher ready')

In [ ]:
# === Load LLM ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === D8 replacement functions ===

def d8a_shuffle_middle(ctx):
    """Shuffle original middle tokens, keep near/far intact."""
    result = list(ctx)
    mid = list(ctx[BRIDGE_SLICE])
    random.shuffle(mid)
    result[BRIDGE_SLICE] = mid
    return result

def d8_replace_middle(ctx, replacement_tokens):
    """Replace middle bridge with given tokens. Must be exactly 20 tokens."""
    assert len(replacement_tokens) == 20, f'Need 20 tokens, got {len(replacement_tokens)}'
    result = list(ctx)
    result[BRIDGE_SLICE] = list(replacement_tokens)
    return result

def find_donor_block(target_doc_id, target_emb, paragraphs, embeddings, doc_mean_embs,
                     mode='same_topic', target_token_start=0, target_token_end=0,
                     context_start=0, rng=None):
    """Find a 20-token donor block.
    mode: 'same_topic' (sim>=0.6), 'diff_topic' (sim<=0.2), 'same_doc' (>=200 tok away)
    """
    if rng is None: rng = random.Random(42)
    
    candidates = []
    target_mean = doc_mean_embs.get(target_doc_id)
    
    for i, (di, did, ps, pe, toks, txt) in enumerate(paragraphs):
        if len(toks) < 20: continue
        
        if mode == 'same_topic':
            if did == target_doc_id: continue
            if target_mean is None: continue
            sim = np.dot(target_mean, embeddings[i]) / (np.linalg.norm(target_mean) * np.linalg.norm(embeddings[i]) + 1e-10)
            if sim >= 0.6: candidates.append((i, toks, sim))
        
        elif mode == 'diff_topic':
            if did == target_doc_id: continue
            if target_mean is None: continue
            sim = np.dot(target_mean, embeddings[i]) / (np.linalg.norm(target_mean) * np.linalg.norm(embeddings[i]) + 1e-10)
            if sim <= 0.2: candidates.append((i, toks, sim))
        
        elif mode == 'same_doc':
            if did != target_doc_id: continue
            # Must be >=200 tokens from target and not overlap context
            if abs(ps - target_token_start) < 200 and abs(pe - target_token_end) < 200: continue
            if ps < target_token_end and pe > context_start: continue  # overlaps context
            candidates.append((i, toks, 0))
    
    if not candidates: return None
    idx, toks, sim = rng.choice(candidates)
    # Take random 20-token slice
    start = rng.randint(0, len(toks) - 20)
    return toks[start:start+20]

@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2: return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full)-1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i+1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0: return float('inf'), float('inf')
    mn = nll/cnt; return math.exp(mn), mn

def compute_corrected_curves(cond_ctx, tgt):
    mc = len(cond_ctx)
    o_ppl, s_ppl = [], []
    for c in range(mc+1):
        pfx = cond_ctx[-c:] if c > 0 else []
        p, _ = ppl_nll(pfx, tgt)
        o_ppl.append(p)
        if c == 0:
            s_ppl.append(p)
        else:
            rng = random.Random(SEED + c)
            sp_l = []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                sp, _ = ppl_nll(sh, tgt)
                if not math.isinf(sp): sp_l.append(sp)
            s_ppl.append(np.mean(sp_l) if sp_l else p)
    dists = list(range(1, mc+1))
    mo = [o_ppl[d-1]-o_ppl[d] for d in dists]
    ms = [s_ppl[d-1]-s_ppl[d] for d in dists]
    delta = [a-b for a,b in zip(mo,ms)]
    return {'distances': dists, 'ordered_ppl': o_ppl, 'delta_ppl': delta}

print('D8 functions ready')

In [ ]:
# === Quality checks ===
print('='*60)
print('QUALITY CHECKS (10 targets per corpus)')
print('='*60)

for cn, cp in CORPORA.items():
    print(f'\n--- {cn} ---')
    docs = []; 
    with open(cp) as f:
        for line in f: docs.append(json.loads(line))
    
    # Build index for this corpus
    print(f'  Building donor index...')
    _, paragraphs, embeddings, doc_mean_embs = build_donor_index(cp, tokenizer)
    print(f'  {len(paragraphs)} paragraphs indexed')
    
    rng_qc = random.Random(42)
    checked = 0
    for doc in rng_qc.sample(docs, min(10, len(docs))):
        fids = tokenizer.encode(doc['text'], add_special_tokens=False)
        n = len(fids)
        ts = int(n * 0.5)
        te = min(ts + TARGET_LEN, n)
        if ts < MIN_BEFORE: continue
        ctx = fids[ts-C:ts]
        tgt = fids[ts:te]
        did = doc.get('doc_id', '')
        
        # D8a
        d8a = d8a_shuffle_middle(ctx)
        assert len(d8a) == C
        assert d8a[FAR_SLICE] == ctx[FAR_SLICE], 'D8a far changed'
        assert d8a[NEAR_SLICE] == ctx[NEAR_SLICE], 'D8a near changed'
        assert Counter(d8a[BRIDGE_SLICE]) == Counter(ctx[BRIDGE_SLICE]), 'D8a bridge multiset'
        
        # D8b
        donor_b = find_donor_block(did, None, paragraphs, embeddings, doc_mean_embs,
                                   'same_topic', rng=rng_qc)
        if donor_b:
            d8b = d8_replace_middle(ctx, donor_b)
            assert len(d8b) == C
            assert d8b[:30] == ctx[:30], 'D8b far changed'
            assert d8b[50:] == ctx[50:], 'D8b near changed'
        
        # D8c
        donor_c = find_donor_block(did, None, paragraphs, embeddings, doc_mean_embs,
                                   'diff_topic', rng=rng_qc)
        if donor_c:
            d8c = d8_replace_middle(ctx, donor_c)
            assert len(d8c) == C
            assert d8c[:30] == ctx[:30] and d8c[50:] == ctx[50:]
        
        # D8d
        donor_d = find_donor_block(did, None, paragraphs, embeddings, doc_mean_embs,
                                   'same_doc', target_token_start=ts,
                                   target_token_end=te, context_start=ts-C, rng=rng_qc)
        if donor_d:
            d8d = d8_replace_middle(ctx, donor_d)
            assert len(d8d) == C
            assert d8d[:30] == ctx[:30] and d8d[50:] == ctx[50:]
        
        # No target leakage
        for name, d8ctx in [('D8a', d8a)] + ([('D8b', d8b)] if donor_b else []) + \
                            ([('D8c', d8c)] if donor_c else []) + ([('D8d', d8d)] if donor_d else []):
            for t in tgt:
                if ctx.count(t) == 0 and t in d8ctx:
                    print(f'  WARNING: {name} target leakage!')
        
        checked += 1
    
    print(f'  {checked} targets checked — all assertions passed')

In [ ]:
# === Run D8 conditions ===

CONDS = ['D8a', 'D8b', 'D8c', 'D8d']
d8_stats = {}

for cn, cp in CORPORA.items():
    print(f'\n{"="*60}')
    print(cn)
    print(f'{"="*60}')
    
    docs = []
    with open(cp) as f:
        for line in f: docs.append(json.loads(line))
    
    # Build donor index
    print('  Building donor index...')
    _, paragraphs, embeddings, doc_mean_embs = build_donor_index(cp, tokenizer)
    
    for cond in CONDS:
        cache = BASE / f'llama_{cn}_{cond}.json'
        if cache.exists():
            with open(cache) as f: n = len(json.load(f))
            print(f'  {cond}: cached ({n})'); continue
        
        t0 = time.time()
        results = []
        n_elig, n_tot = 0, 0
        rng_d = random.Random(SEED)
        
        for doc in tqdm(docs, desc=f'{cn}/{cond}'):
            fids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(fids)
            did = doc.get('doc_id', '')
            
            for frac in TARGET_FRACS:
                ts = int(n * frac)
                te = min(ts + TARGET_LEN, n)
                if ts < MIN_BEFORE or te - ts < 5: continue
                ctx = fids[ts-C:ts]
                tgt = fids[ts:te]
                n_tot += 1
                
                if cond == 'D8a':
                    cond_ctx = d8a_shuffle_middle(ctx)
                    n_elig += 1
                elif cond == 'D8b':
                    donor = find_donor_block(did, None, paragraphs, embeddings,
                                            doc_mean_embs, 'same_topic', rng=rng_d)
                    if donor is None: continue
                    cond_ctx = d8_replace_middle(ctx, donor)
                    n_elig += 1
                elif cond == 'D8c':
                    donor = find_donor_block(did, None, paragraphs, embeddings,
                                            doc_mean_embs, 'diff_topic', rng=rng_d)
                    if donor is None: continue
                    cond_ctx = d8_replace_middle(ctx, donor)
                    n_elig += 1
                elif cond == 'D8d':
                    donor = find_donor_block(did, None, paragraphs, embeddings,
                                            doc_mean_embs, 'same_doc',
                                            target_token_start=ts, target_token_end=te,
                                            context_start=ts-C, rng=rng_d)
                    if donor is None: continue
                    cond_ctx = d8_replace_middle(ctx, donor)
                    n_elig += 1
                
                assert len(cond_ctx) == C
                assert cond_ctx[FAR_SLICE] == ctx[FAR_SLICE], f'{cond} far changed'
                assert cond_ctx[NEAR_SLICE] == ctx[NEAR_SLICE], f'{cond} near changed'
                
                r = compute_corrected_curves(cond_ctx, tgt)
                r['doc_id'] = did
                r['target_frac'] = frac
                results.append(r)
        
        with open(cache, 'w') as f: json.dump(results, f)
        elapsed = time.time() - t0
        elig = n_elig/n_tot if n_tot > 0 else 0
        d8_stats[(cn, cond)] = {'elig': elig, 'n': n_elig, 'total': n_tot}
        print(f'  {cond}: {len(results)} results in {elapsed/60:.1f} min')
        print(f'  Eligible: {n_elig}/{n_tot} ({elig:.0%})')
        if results:
            print(f'  Mean corrected Δ: {np.mean([np.mean(r["delta_ppl"]) for r in results]):.6f}')

In [ ]:
# === Results table ===
import matplotlib.pyplot as plt

print(f'\n{"="*90}')
print('D8 MIDDLE-BRIDGE REPLACEMENT — Corrected Marginals')
print(f'{"="*90}')

all_conds = ['intact', 'D8a', 'D8b', 'D8c', 'D8d']

for cn in CORPORA:
    print(f'\n--- {cn} ---')
    print(f'{"Cond":<12} {"TotalΔ":>8} {"NearΔ":>8} {"BridgeΔ":>8} {"FarΔ":>8} '
          f'{"BridgeChg":>10} {"NearChg":>10} {"FarChg":>10} {"Redist":>8}')
    print('-' * 95)
    
    intact_bands = None
    
    for cond in all_conds:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists():
            print(f'{cond:<12} {"—":>8}'); continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        total = np.mean(curve)
        near_d = np.mean(curve[:50])    # d=1–50
        bridge_d = np.mean(curve[50:70]) # d=51–70
        far_d = np.mean(curve[70:])      # d=71–100
        
        if cond == 'intact':
            intact_bands = (total, near_d, bridge_d, far_d)
            print(f'{cond:<12} {total:>8.4f} {near_d:>8.4f} {bridge_d:>8.4f} {far_d:>8.4f} '
                  f'{"—":>10} {"—":>10} {"—":>10} {"—":>8}')
        elif intact_bands:
            i_tot, i_near, i_bridge, i_far = intact_bands
            bc = bridge_d - i_bridge
            nc = near_d - i_near
            fc = far_d - i_far
            redist = abs(nc) + abs(fc)
            print(f'{cond:<12} {total:>8.4f} {near_d:>8.4f} {bridge_d:>8.4f} {far_d:>8.4f} '
                  f'{bc:>10.4f} {nc:>10.4f} {fc:>10.4f} {redist:>8.4f}')
    print()

# Topic decomposition
print('\nTopic/Document Decomposition:')
for cn in CORPORA:
    bridges = {}
    for cond in all_conds:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if r: bridges[cond] = np.mean([np.mean(x['delta_ppl'][50:70]) for x in r])
    
    if all(c in bridges for c in ['intact', 'D8a', 'D8b', 'D8c', 'D8d']):
        print(f'  {cn}:')
        print(f'    Order contribution:    intact - D8a = {bridges["intact"]-bridges["D8a"]:.4f}')
        print(f'    Topic contribution:    D8b - D8c    = {bridges["D8b"]-bridges["D8c"]:.4f}')
        print(f'    Document contribution: D8d - D8b    = {bridges["D8d"]-bridges["D8b"]:.4f}')
        print(f'    Lexical contribution:  D8a - D8c    = {bridges["D8a"]-bridges["D8c"]:.4f}')

In [ ]:
# === Plots ===

colors = {'intact': 'blue', 'D8a': 'orange', 'D8b': 'green', 'D8c': 'red', 'D8d': 'purple'}

# 1. Corrected marginal curves
fig, axes = plt.subplots(1, len(CORPORA), figsize=(7*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]
for idx, cn in enumerate(CORPORA):
    ax = axes[idx]
    for cond in all_conds:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        curve = np.mean([x['delta_ppl'] for x in r], axis=0)
        ax.plot(range(1, len(curve)+1), uniform_filter1d(curve, 5),
                color=colors[cond], linewidth=2, label=cond)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.axvspan(51, 70, alpha=0.1, color='yellow', label='bridge')
    ax.set_title(cn, fontweight='bold')
    ax.set_xlabel('Distance d'); ax.set_ylabel('Corrected Marginal Δ')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.15)
plt.suptitle('D8: Middle-Bridge Replacement — Corrected Marginals', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D8_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Band bar plot
fig, axes = plt.subplots(1, len(CORPORA), figsize=(8*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]
for idx, cn in enumerate(CORPORA):
    ax = axes[idx]
    x = np.arange(len(all_conds))
    w = 0.25
    near_vals, bridge_vals, far_vals = [], [], []
    valid_conds = []
    for cond in all_conds:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        curve = np.mean([x_['delta_ppl'] for x_ in r], axis=0)
        near_vals.append(np.mean(curve[:50]))
        bridge_vals.append(np.mean(curve[50:70]))
        far_vals.append(np.mean(curve[70:]))
        valid_conds.append(cond)
    x = np.arange(len(valid_conds))
    ax.bar(x - w, near_vals, w, color='#2196F3', alpha=0.7, label='Near d=1–50')
    ax.bar(x, bridge_vals, w, color='#FF9800', alpha=0.7, label='Bridge d=51–70')
    ax.bar(x + w, far_vals, w, color='#4CAF50', alpha=0.7, label='Far d=71–100')
    ax.set_xticks(x); ax.set_xticklabels(valid_conds, fontsize=10)
    ax.set_ylabel('Mean Corrected Δ'); ax.set_title(cn, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.15, axis='y')
plt.suptitle('D8: Near/Bridge/Far Δ by Condition', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D8_bands.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Redistribution plot
fig, axes = plt.subplots(1, len(CORPORA), figsize=(7*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]
for idx, cn in enumerate(CORPORA):
    ax = axes[idx]
    ip = BASE / f'llama_{cn}_intact.json'
    if not ip.exists(): continue
    with open(ip) as f: ir = json.load(f)
    ic = np.mean([x['delta_ppl'] for x in ir], axis=0)
    i_near, i_far = np.mean(ic[:50]), np.mean(ic[70:])
    
    cond_labels, near_chg, far_chg = [], [], []
    for cond in ['D8a', 'D8b', 'D8c', 'D8d']:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        curve = np.mean([x['delta_ppl'] for x in r], axis=0)
        cond_labels.append(cond)
        near_chg.append(np.mean(curve[:50]) - i_near)
        far_chg.append(np.mean(curve[70:]) - i_far)
    
    x = np.arange(len(cond_labels))
    ax.bar(x - 0.15, near_chg, 0.3, color='#2196F3', alpha=0.7, label='NearChange')
    ax.bar(x + 0.15, far_chg, 0.3, color='#4CAF50', alpha=0.7, label='FarChange')
    ax.set_xticks(x); ax.set_xticklabels(cond_labels)
    ax.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax.set_ylabel('Change from Intact'); ax.set_title(cn, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.15, axis='y')
plt.suptitle('D8: Redistribution — Change in Near/Far when Bridge is Replaced',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D8_redistribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('All figures saved')